# Mini-TP 6 · Tu modelo en el Data Lake (individual)

**Operaciones de Aprendizaje Automático II · CEIA – FIUBA**

## Consigna
Sube el artefacto de **tu propio modelo** a un Data Lake (MinIO/S3) y **sírvelo cargándolo
desde allí**, no desde disco. Organiza el bucket con **zonas** y **versión**.

### Qué se entrega
1. El bucket con zonas (`raw`, `curated`, `models`) y tu modelo en `models/v1/…`.
2. Una función que **carga el modelo desde el lake** y predice.
3. (Opcional) El experimento registrado con **MLflow apuntando a MinIO/S3**.
4. Una reflexión breve: ¿por qué servir desde el lake y no hornear el modelo en la imagen?

### Se evalúa
- Que **corra de punta a punta**.
- Uso correcto de la API S3 (`boto3`): bucket, zonas por prefijo, versión del modelo.
- Que el modelo se **cargue desde el lake** para servir.
- (Opcional) Integración con MLflow.

> Antes de empezar, levanta MinIO con Docker (ver la *Guía de la práctica* o el tutorial
> `Teoria/datalake_tutorial.ipynb`) y instala las dependencias:
> `uv add boto3 pandas pyarrow scikit-learn mlflow joblib`


## 0. Configuración (dada)
Credenciales por variables de entorno y endpoint de MinIO. No modifiques salvo que tu
MinIO use otro puerto/credenciales.


In [ ]:
import os
os.environ.setdefault("AWS_ACCESS_KEY_ID", "minio")
os.environ.setdefault("AWS_SECRET_ACCESS_KEY", "minio123")
os.environ.setdefault("AWS_DEFAULT_REGION", "us-east-1")

ENDPOINT = "http://localhost:9000"
BUCKET = "datalake-tp6"      # usa tu propio bucket
print("Endpoint:", ENDPOINT, "| Bucket:", BUCKET)

In [ ]:
import boto3
from botocore.client import Config

s3 = boto3.client(
    "s3",
    endpoint_url=ENDPOINT,
    aws_access_key_id=os.environ["AWS_ACCESS_KEY_ID"],
    aws_secret_access_key=os.environ["AWS_SECRET_ACCESS_KEY"],
    region_name=os.environ.get("AWS_DEFAULT_REGION", "us-east-1"),
    config=Config(signature_version="s3v4"),
)
print("Cliente S3 creado para", ENDPOINT)


## 1. Crear el bucket y las zonas


In [ ]:
def crear_bucket_y_zonas(bucket):
    existentes = {b["Name"] for b in s3.list_buckets().get("Buckets", [])}
    if bucket not in existentes:
        s3.create_bucket(Bucket=bucket)
        print("Bucket creado:", bucket)
    else:
        print("Bucket existente:", bucket)

    for zona in ("raw", "curated", "models"):
        key = f"{zona}/.keep"
        s3.put_object(Bucket=bucket, Key=key, Body=b"")
        print("Zona inicializada:", key)

crear_bucket_y_zonas(BUCKET)


## 2. Entrenar (o cargar) tu modelo
Usa **tu modelo** del curso. Si aún no lo tienes a mano, un modelo simple de scikit-learn
sirve para completar el flujo.


In [ ]:
import io, joblib
import numpy as np
import pandas as pd
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

# Dataset pequeño y público para dejar el flujo autocontenido.
iris = load_iris(as_frame=True)
df_raw = iris.frame.copy()
X = iris.data
y = iris.target
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

modelo = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", LogisticRegression(max_iter=500, random_state=42)),
])
modelo.fit(X_train, y_train)
acc_test = accuracy_score(y_test, modelo.predict(X_test))

# Curated: ejemplo de dataset validado/normalizado, manteniendo target y nombres de columnas.
scaler_curated = StandardScaler().fit(X_train)
df_curated = pd.DataFrame(scaler_curated.transform(X), columns=X.columns)
df_curated["target"] = y.to_numpy()

print(f"Modelo entrenado. Accuracy test = {acc_test:.3f}")
print("Raw:", df_raw.shape, "| Curated:", df_curated.shape)


## 3. Subir dataset y modelo al lake


In [ ]:
# 1) RAW -> CSV en memoria
raw_buffer = io.StringIO()
df_raw.to_csv(raw_buffer, index=False)
RAW_KEY = "raw/iris/iris.csv"
s3.put_object(Bucket=BUCKET, Key=RAW_KEY, Body=raw_buffer.getvalue().encode("utf-8"), ContentType="text/csv")

# 2) CURATED -> Parquet en memoria
curated_buffer = io.BytesIO()
df_curated.to_parquet(curated_buffer, index=False)
CURATED_KEY = "curated/iris/iris.parquet"
s3.put_object(Bucket=BUCKET, Key=CURATED_KEY, Body=curated_buffer.getvalue(), ContentType="application/octet-stream")

# 3) MODELO -> joblib en memoria y versión v1
model_buffer = io.BytesIO()
joblib.dump(modelo, model_buffer)
MODEL_KEY = "models/v1/iris_logreg.pkl"
s3.put_object(Bucket=BUCKET, Key=MODEL_KEY, Body=model_buffer.getvalue(), ContentType="application/octet-stream")

print("Objetos subidos:")
for key in (RAW_KEY, CURATED_KEY, MODEL_KEY):
    print(" -", f"s3://{BUCKET}/{key}")


## 4. Servir el modelo desde el lake
El corazón del Mini-TP: **cargar el modelo desde el lake** (no desde disco) y predecir.


In [ ]:
def cargar_modelo_desde_lake(bucket, key):
    respuesta = s3.get_object(Bucket=bucket, Key=key)
    contenido = respuesta["Body"].read()
    return joblib.load(io.BytesIO(contenido))

modelo_lake = cargar_modelo_desde_lake(BUCKET, MODEL_KEY)

ejemplo = X_test.iloc[[0]]
pred = modelo_lake.predict(ejemplo)[0]
proba = modelo_lake.predict_proba(ejemplo)[0]

print("Ejemplo:", ejemplo.to_dict(orient="records")[0])
print("Predicción desde modelo cargado del lake:", int(pred))
print("Probabilidades:", np.round(proba, 4))


## 5. (Opcional) MLflow apuntando a MinIO/S3


In [ ]:
import os, tempfile
import mlflow

os.environ["MLFLOW_S3_ENDPOINT_URL"] = ENDPOINT
mlflow.set_tracking_uri("sqlite:///mlflow_tp6.db")

EXPERIMENTO = "mini_tp6_datalake"
exp = mlflow.get_experiment_by_name(EXPERIMENTO)
if exp is None:
    exp_id = mlflow.create_experiment(
        EXPERIMENTO,
        artifact_location=f"s3://{BUCKET}/mlflow"
    )
else:
    exp_id = exp.experiment_id

with mlflow.start_run(experiment_id=exp_id) as run:
    mlflow.log_param("modelo", "LogisticRegression")
    mlflow.log_param("dataset", "iris")
    mlflow.log_param("version", "v1")
    mlflow.log_metric("accuracy_test", float(acc_test))

    with tempfile.TemporaryDirectory() as tmpdir:
        local_model = os.path.join(tmpdir, "iris_logreg.pkl")
        joblib.dump(modelo, local_model)
        mlflow.log_artifact(local_model, artifact_path="model")

    print("MLflow run:", run.info.run_id)
    print("Artifact URI:", mlflow.get_artifact_uri())


## 6. Reflexión

1. Conviene que el servicio **cargue el modelo desde el lake** porque desacopla el artefacto de ML de la imagen del contenedor. Así se puede actualizar o hacer rollback del modelo sin reconstruir y desplegar toda la aplicación, y además se mantiene una fuente central y auditable de artefactos.
2. Publicaría cada modelo en una ruta versionada (`models/v1/...`, `models/v2/...`) y haría que producción apunte explícitamente a una versión validada. El cambio a `v2` se haría después de pruebas, conservando `v1` para rollback inmediato.
3. Guardaría las predicciones en **`curated`** (por ejemplo `curated/predictions/...`) porque ya son datos derivados/procesados y con un esquema conocido. `raw` lo reservaría para entradas sin transformar y `models` exclusivamente para artefactos de modelos.
